# Traffic Congestion Analytics - Exploratory Data Analysis

**Project**: Traffic Congestion Analytics System  
**Dataset**: Highway Traffic Volume from Kaggle  
**Objective**: Analyze traffic patterns, identify congestion, and build predictive models

---

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

print("Libraries imported successfully!")

## 2. Load Dataset from Kaggle

In [ ]:
# Download dataset using kagglehub
import kagglehub

path = kagglehub.download("galenchen/highway-traffic-volume")
print("Path to dataset files:", path)

# Find CSV file
csv_files = list(Path(path).glob("*.csv"))
print(f"\nFound {len(csv_files)} CSV file(s)")
print(f"Loading: {csv_files[0]}")

# Load data
df = pd.read_csv(csv_files[0])
print(f"\nData loaded successfully!")
print(f"Shape: {df.shape}")

## 3. Initial Data Exploration

In [ ]:
# Display first few rows
print("First 5 rows:")
df.head()

In [ ]:
# Data info
print("Dataset Information:")
df.info()

In [ ]:
# Statistical summary
print("Statistical Summary:")
df.describe()

In [ ]:
# Check for missing values
print("Missing Values:")
missing = df.isnull().sum()
print(missing[missing > 0])

print(f"\nTotal missing values: {df.isnull().sum().sum()}")

## 4. Data Cleaning

In [ ]:
# Remove null values
print(f"Original shape: {df.shape}")
df_clean = df.dropna()
print(f"After removing nulls: {df_clean.shape}")
print(f"Removed {df.shape[0] - df_clean.shape[0]} rows")

# Convert date_time to datetime
df_clean['date_time'] = pd.to_datetime(df_clean['date_time'])
print(f"\nDate range: {df_clean['date_time'].min()} to {df_clean['date_time'].max()}")

## 5. Feature Engineering

In [ ]:
# Extract time-based features
df_clean['hour'] = df_clean['date_time'].dt.hour
df_clean['day'] = df_clean['date_time'].dt.day
df_clean['month'] = df_clean['date_time'].dt.month
df_clean['year'] = df_clean['date_time'].dt.year
df_clean['weekday'] = df_clean['date_time'].dt.weekday
df_clean['weekday_name'] = df_clean['date_time'].dt.day_name()

# Weekend flag
df_clean['weekend_flag'] = (df_clean['weekday'] >= 5).astype(int)

# Congestion flag (threshold: 4500)
CONGESTION_THRESHOLD = 4500
df_clean['is_congested'] = (df_clean['traffic_volume'] > CONGESTION_THRESHOLD).astype(int)

print("Engineered features:")
print(df_clean[['date_time', 'hour', 'weekday_name', 'weekend_flag', 'is_congested']].head())

## 6. Traffic Volume Analysis

In [ ]:
# Traffic volume distribution
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.hist(df_clean['traffic_volume'], bins=50, color='#FF6B00', edgecolor='black', alpha=0.7)
plt.axvline(CONGESTION_THRESHOLD, color='red', linestyle='--', linewidth=2, label=f'Threshold ({CONGESTION_THRESHOLD})')
plt.xlabel('Traffic Volume')
plt.ylabel('Frequency')
plt.title('Traffic Volume Distribution')
plt.legend()

plt.subplot(1, 2, 2)
plt.boxplot(df_clean['traffic_volume'], vert=True, patch_artist=True,
            boxprops=dict(facecolor='#FF6B00', alpha=0.7))
plt.ylabel('Traffic Volume')
plt.title('Traffic Volume Box Plot')

plt.tight_layout()
plt.show()

print(f"\nTraffic Volume Statistics:")
print(f"Mean: {df_clean['traffic_volume'].mean():.2f}")
print(f"Median: {df_clean['traffic_volume'].median():.2f}")
print(f"Std Dev: {df_clean['traffic_volume'].std():.2f}")

## 7. Hourly Traffic Analysis

In [ ]:
# Average traffic by hour
hourly_avg = df_clean.groupby('hour')['traffic_volume'].mean()

plt.figure(figsize=(14, 6))
plt.plot(hourly_avg.index, hourly_avg.values, marker='o', linewidth=2.5, 
         markersize=8, color='#FF6B00', label='Average Traffic')
plt.fill_between(hourly_avg.index, hourly_avg.values, alpha=0.3, color='#FF6B00')
plt.xlabel('Hour of Day')
plt.ylabel('Average Traffic Volume')
plt.title('Average Traffic Volume by Hour of Day')
plt.xticks(range(24))
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print(f"\nPeak Traffic Hour: {hourly_avg.idxmax()}:00 with {hourly_avg.max():.0f} vehicles")
print(f"Lowest Traffic Hour: {hourly_avg.idxmin()}:00 with {hourly_avg.min():.0f} vehicles")

## 8. Weekly Traffic Patterns

In [ ]:
# Average traffic by day of week
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily_avg = df_clean.groupby('weekday_name')['traffic_volume'].mean().reindex(days_order)

plt.figure(figsize=(12, 6))
colors = ['#FF6B00' if day not in ['Saturday', 'Sunday'] else '#2ECC71' for day in days_order]
bars = plt.bar(range(len(daily_avg)), daily_avg.values, color=colors, edgecolor='black', linewidth=1.2)
plt.xlabel('Day of Week')
plt.ylabel('Average Traffic Volume')
plt.title('Average Traffic Volume by Day of Week')
plt.xticks(range(len(days_order)), days_order, rotation=45)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f"\nBusiest Day: {daily_avg.idxmax()} with {daily_avg.max():.0f} vehicles")
print(f"Quietest Day: {daily_avg.idxmin()} with {daily_avg.min():.0f} vehicles")

## 9. Monthly Traffic Trends

In [ ]:
# Average traffic by month
monthly_avg = df_clean.groupby('month')['traffic_volume'].mean()
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

plt.figure(figsize=(14, 6))
plt.plot(monthly_avg.index, monthly_avg.values, marker='s', linewidth=2.5, 
         markersize=10, color='#FF6B00', label='Average Traffic')
plt.fill_between(monthly_avg.index, monthly_avg.values, alpha=0.2, color='#FF6B00')
plt.xlabel('Month')
plt.ylabel('Average Traffic Volume')
plt.title('Average Traffic Volume by Month')
plt.xticks(range(1, 13), month_names)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print(f"\nHighest Traffic Month: {month_names[monthly_avg.idxmax()-1]} with {monthly_avg.max():.0f} vehicles")
print(f"Lowest Traffic Month: {month_names[monthly_avg.idxmin()-1]} with {monthly_avg.min():.0f} vehicles")

## 10. Weather Impact Analysis

In [ ]:
# Traffic by weather condition
weather_avg = df_clean.groupby('weather_main')['traffic_volume'].mean().sort_values(ascending=False)

plt.figure(figsize=(12, 6))
colors_weather = plt.cm.RdYlGn_r(np.linspace(0.3, 0.7, len(weather_avg)))
plt.barh(range(len(weather_avg)), weather_avg.values, color=colors_weather, edgecolor='black', linewidth=1.2)
plt.xlabel('Average Traffic Volume')
plt.ylabel('Weather Condition')
plt.title('Traffic Volume by Weather Condition')
plt.yticks(range(len(weather_avg)), weather_avg.index)
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("\nTraffic by Weather:")
print(weather_avg)

## 11. Congestion Analysis

In [ ]:
# Congestion distribution
congestion_counts = df_clean['is_congested'].value_counts()
congestion_rate = (congestion_counts[1] / len(df_clean)) * 100

plt.figure(figsize=(10, 6))
colors_pie = ['#2ECC71', '#FF6B00']
plt.pie(congestion_counts.values, labels=['Normal Traffic', 'Congested'], 
        autopct='%1.1f%%', startangle=90, colors=colors_pie,
        textprops={'fontsize': 12, 'fontweight': 'bold'}, explode=(0, 0.05))
plt.title('Traffic Congestion Distribution')
plt.axis('equal')
plt.tight_layout()
plt.show()

print(f"\nCongestion Rate: {congestion_rate:.2f}%")
print(f"Normal Traffic: {congestion_counts[0]:,} records")
print(f"Congested: {congestion_counts[1]:,} records")

In [ ]:
# Hourly congestion rate
hourly_congestion = df_clean.groupby('hour')['is_congested'].mean() * 100

plt.figure(figsize=(14, 6))
plt.bar(hourly_congestion.index, hourly_congestion.values, color='#FF6B00', edgecolor='black', linewidth=1.2)
plt.xlabel('Hour of Day')
plt.ylabel('Congestion Rate (%)')
plt.title('Congestion Rate by Hour')
plt.xticks(range(24))
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f"\nPeak Congestion Hour: {hourly_congestion.idxmax()}:00 with {hourly_congestion.max():.1f}% congestion")

## 12. Correlation Analysis

In [ ]:
# Correlation matrix
numeric_cols = ['traffic_volume', 'temp', 'rain_1h', 'snow_1h', 'clouds_all', 
                'hour', 'weekday', 'month', 'weekend_flag']
correlation_matrix = df_clean[numeric_cols].corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, mask=mask, annot=True, fmt='.2f', 
            cmap='RdYlGn', center=0, square=True, linewidths=1,
            cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

print("\nTop Correlations with Traffic Volume:")
traffic_corr = correlation_matrix['traffic_volume'].sort_values(ascending=False)
print(traffic_corr)

## 13. Machine Learning - Linear Regression

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Prepare data
X = df_clean[['hour', 'month', 'weekday', 'weekend_flag', 'temp', 'rain_1h', 'snow_1h', 'clouds_all']]
y = df_clean['traffic_volume']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = LinearRegression()
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Evaluate
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Linear Regression Model Performance:")
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R² Score: {r2:.4f}")

In [ ]:
# Visualize predictions
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.scatter(y_test, y_pred, alpha=0.3, color='#FF6B00')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2)
plt.xlabel('Actual Traffic Volume')
plt.ylabel('Predicted Traffic Volume')
plt.title('Actual vs Predicted Traffic Volume')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
residuals = y_test - y_pred
plt.hist(residuals, bins=50, color='#2ECC71', edgecolor='black', alpha=0.7)
plt.xlabel('Residuals')
plt.ylabel('Frequency')
plt.title('Residuals Distribution')
plt.axvline(0, color='red', linestyle='--', linewidth=2)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 14. Clustering Analysis (K-Means)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Prepare data for clustering
cluster_features = df_clean[['hour', 'traffic_volume', 'temp', 'weekday']].copy()

# Standardize features
scaler = StandardScaler()
cluster_features_scaled = scaler.fit_transform(cluster_features)

# K-Means clustering
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df_clean['cluster'] = kmeans.fit_predict(cluster_features_scaled)

print(f"Clustering Results:")
print(f"Number of clusters: 4")
print(f"Inertia: {kmeans.inertia_:.2f}")
print(f"\nCluster distribution:")
print(df_clean['cluster'].value_counts().sort_index())

In [ ]:
# Visualize clusters
plt.figure(figsize=(14, 8))
colors_cluster = ['#FF6B00', '#2ECC71', '#3498DB', '#9B59B6']

for cluster in range(4):
    cluster_data = df_clean[df_clean['cluster'] == cluster]
    plt.scatter(cluster_data['hour'], cluster_data['traffic_volume'], 
               c=colors_cluster[cluster], label=f'Cluster {cluster}',
               alpha=0.5, s=30, edgecolors='black', linewidth=0.3)

plt.xlabel('Hour of Day')
plt.ylabel('Traffic Volume')
plt.title('Traffic Pattern Clustering (K-Means)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 15. Key Insights Summary

In [ ]:
print("="*60)
print("KEY INSIGHTS - TRAFFIC CONGESTION ANALYTICS")
print("="*60)

print(f"\n📊 DATASET OVERVIEW:")
print(f"   Total Records: {len(df_clean):,}")
print(f"   Date Range: {df_clean['date_time'].min().date()} to {df_clean['date_time'].max().date()}")

print(f"\n🚗 TRAFFIC VOLUME:")
print(f"   Average: {df_clean['traffic_volume'].mean():.0f} vehicles")
print(f"   Peak: {df_clean['traffic_volume'].max():,} vehicles")
print(f"   Minimum: {df_clean['traffic_volume'].min():,} vehicles")

print(f"\n⏰ PEAK HOURS:")
peak_hour = hourly_avg.idxmax()
print(f"   Peak Traffic Hour: {peak_hour}:00")
print(f"   Peak Traffic Volume: {hourly_avg.max():.0f} vehicles")

print(f"\n📅 WEEKLY PATTERNS:")
busiest_day = daily_avg.idxmax()
print(f"   Busiest Day: {busiest_day}")
print(f"   Average Volume: {daily_avg.max():.0f} vehicles")

print(f"\n⚠️ CONGESTION:")
print(f"   Congestion Threshold: {CONGESTION_THRESHOLD} vehicles")
print(f"   Congestion Rate: {congestion_rate:.2f}%")
print(f"   Peak Congestion Hour: {hourly_congestion.idxmax()}:00")

print(f"\n🌤️ WEATHER IMPACT:")
print(f"   Highest Traffic Weather: {weather_avg.idxmax()}")
print(f"   Average Volume: {weather_avg.max():.0f} vehicles")

print(f"\n🤖 MODEL PERFORMANCE:")
print(f"   Algorithm: Linear Regression")
print(f"   MAE: {mae:.2f}")
print(f"   RMSE: {rmse:.2f}")
print(f"   R² Score: {r2:.4f}")

print("\n" + "="*60)
print("Analysis Complete!")
print("="*60)

## Conclusion

This exploratory data analysis has revealed:

1. **Peak Traffic**: Occurs during morning (7-9 AM) and evening (4-6 PM) rush hours
2. **Weekly Patterns**: Weekdays show higher traffic than weekends
3. **Seasonal Trends**: Traffic volumes vary throughout the year
4. **Weather Impact**: Different weather conditions affect traffic flow
5. **Congestion**: Significant portion of traffic exceeds threshold during peak hours
6. **Predictive Model**: Linear Regression achieves reasonable accuracy for volume prediction
7. **Clustering**: K-Means identifies distinct traffic pattern groups

**Recommendations for Urban Planners:**
- Implement dynamic traffic management during peak hours (7-9 AM, 4-6 PM)
- Focus congestion mitigation on weekdays
- Consider weather-based traffic control strategies
- Use predictive models for proactive traffic management

---
**End of Analysis**